#  moving average

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

In [3]:
from zipline.api import (
    attach_pipeline,
    date_rules,
    time_rules,
    order_target_percent,
    pipeline_output,
    record,
    schedule_function,
    get_open_orders,
    calendars
)
from zipline import run_algorithm
from zipline.finance import commission, slippage
from zipline.pipeline import Pipeline, CustomFactor
from zipline.pipeline.factors import Returns, AverageDollarVolume
import pyfolio as pf
import numpy as np
import pandas as pd
from pathlib import Path

In [4]:
MONTH = 21
YEAR = 12 * MONTH
N_LONGS = N_SHORTS = 25
VOL_SCREEN = 1000

In [5]:
class MeanReversion(CustomFactor):
    """Compute ratio of latest monthly return to 12m average,
       normalized by std dev of monthly returns"""
    inputs = [Returns(window_length=MONTH)]
    window_length = YEAR

    def compute(self, today, assets, out, monthly_returns):
        df = pd.DataFrame(monthly_returns)
        out[:] = df.iloc[-1].sub(df.mean()).div(df.std())


def compute_factors():
    """Create factor pipeline incl. mean reversion,
        filtered by 30d Dollar Volume; capture factor ranks"""
    mean_reversion = MeanReversion()
    dollar_volume = AverageDollarVolume(window_length=20)
    return Pipeline(columns={'longs': mean_reversion.bottom(N_LONGS),
                             'shorts': mean_reversion.top(N_SHORTS),
                             'ranking': mean_reversion.rank(ascending=False)},
                    screen=dollar_volume.top(VOL_SCREEN))


def exec_trades(data, assets, target_percent):
    """Place orders for assets using target portfolio percentage"""
    for asset in assets:
        if data.can_trade(asset) and not get_open_orders(asset):
            order_target_percent(asset, target_percent)



def rebalance(context, data):
    """Compute long, short and obsolete holdings; place trade orders"""
    factor_data = context.factor_data
    record(factor_data=factor_data.ranking)

    assets = factor_data.index
    record(prices=data.current(assets, 'price'))

    longs = assets[factor_data.longs]
    shorts = assets[factor_data.shorts]
    divest = set(context.portfolio.positions.keys()) - set(longs.union(shorts))

    exec_trades(data, assets=divest, target_percent=0)
    exec_trades(data, assets=longs, target_percent=1 / N_LONGS)
    exec_trades(data, assets=shorts, target_percent=-1 / N_SHORTS)


def initialize(context):
    """Setup: register pipeline, schedule rebalancing,
        and set trading params"""
    attach_pipeline(compute_factors(), 'factor_pipeline')
    schedule_function(rebalance,
                      date_rule = date_rules.month_start(), time_rule=time_rules.market_close(), half_days=True, calendar=None)
    
    context.set_commission(commission.PerShare(cost=.01, min_trade_cost=0))
    context.set_slippage(slippage.VolumeShareSlippage())


def before_trading_start(context, data):
    """Run factor pipeline"""
    context.factor_data = pipeline_output('factor_pipeline')

def analyze(context, perf):
    returns, postions, transactions = pf.utils.extract_rets_pos_txn_from_zipline(perf) #수익률, 포지션, 거래 반환
    pf.create_returns_tear_sheet(returns, benchmark_rets=None)

def analyze(context, perf):
    returns, positions, transactions = pf.utils.extract_rets_pos_txn_from_zipline(perf)
    benchmark_rets = etf_prces[start_date:end_date]['SPY'].pct_change().dropna()
    pf.create_returns_tear_sheet(returns, benchmark_rets=benchmark_rets)


In [11]:
DATA_STORE = Path('E:/mlft/data/assets.h5')
print(DATA_STORE.cwd())

key = f'us/etfs/prices/yfinance'
with pd.HDFStore(DATA_STORE, mode='r') as store:
    etf =store[key]
etf = etf[~etf.index.duplicated()]
etf_prces = etf['close'].unstack('symbol')
# etf_prces.head()

C:\Users\taest\Documents\github\research_market_finance\note\backtest\2_modeling


In [12]:
etf_prces.index = etf_prces.index.map(lambda x: pd.Timestamp(x).tz_localize('UTC'))

In [13]:
start_date = pd.Timestamp('2010-01-01').tz_localize('UTC')
end_date = pd.Timestamp('2024-07-01').tz_localize('UTC')

In [14]:
# Run algorithm
result = run_algorithm(
    start = start_date,
    end = end_date,
    initialize=initialize,
    before_trading_start=before_trading_start,
    analyze=analyze,
    capital_base=100000,
    data_frequency='daily',
    benchmark_returns=etf_prces[start_date:end_date]['SPY'].pct_change().dropna(),
    bundle='us_etfs'
)

AttributeError: 'datetime.timezone' object has no attribute 'key'

In [ ]:
benchmark_rets = etf_prces[start_date:end_date]['SPY'].pct_change().dropna()
pf.create_returns_tear_sheet(returns, benchmark_rets=benchmark_rets)

---